# Tutorial 05 — Structure-Tensor Orientation Extraction

This notebook computes orientation fields from the local source package and does not load committed result images.

In [ ]:
LANGUAGE = "en"

import importlib.util
import os
import sys
from pathlib import Path


def _is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (
        path / "src" / "biomechanics_tutorials"
    ).is_dir()


def _find_repository_root() -> Path:
    candidates = []
    installed_spec = importlib.util.find_spec("biomechanics_tutorials")
    if installed_spec is not None and installed_spec.origin:
        package_file = Path(installed_spec.origin).resolve()
        if len(package_file.parents) >= 3:
            candidates.append(package_file.parents[2])
    environment_root = os.environ.get("BMRT_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    home = Path.home()
    for directory in [home, home / "Desktop", home / "Documents", home / "Downloads"]:
        candidates.append(directory / "Biomechanics-Research-Tutorials")
        if directory.is_dir():
            candidates.extend(directory.glob("Biomechanics-Research-Tutorials*"))
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_repository_root(candidate):
            return candidate
    raise RuntimeError(
        "Repository root was not found. Extract the complete repository or install it with python -m pip install -e .[dev]"
    )


REPOSITORY_ROOT = _find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import numpy as np
import matplotlib.pyplot as plt
import biomechanics_tutorials
from biomechanics_tutorials.structure_tensor import (
    StructureTensorParameters,
    confidence_mask,
    orientation_error_metrics,
    structure_tensor_orientation,
    synthetic_crossing_fibers,
    synthetic_curved_fibers,
    synthetic_parallel_fibers,
    weighted_axial_mean,
)

print("Repository:", REPOSITORY_ROOT)
print("Python:", sys.executable)
print("Package:", Path(biomechanics_tutorials.__file__).resolve())

## Baseline field

In [ ]:
image, truth = synthetic_parallel_fibers(
    angle_degrees=30.0,
    noise_std=0.06,
    seed=5,
)
parameters = StructureTensorParameters(gradient_sigma=1.0, integration_sigma=3.0)
result = structure_tensor_orientation(image, parameters)
mask = confidence_mask(result, coherence_threshold=0.4, energy_quantile=0.15, border=10)
metrics = orientation_error_metrics(result["orientation"], truth, mask)
metrics

## Orientation maps

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Image" if LANGUAGE == "en" else "Изображение")
axes[1].imshow(np.rad2deg(truth), cmap="twilight", vmin=-90, vmax=90)
axes[1].set_title("Truth" if LANGUAGE == "en" else "Истина")
axes[2].imshow(np.where(mask, np.rad2deg(result["orientation"]), np.nan), cmap="twilight", vmin=-90, vmax=90)
axes[2].set_title("Estimate" if LANGUAGE == "en" else "Оценка")
axes[3].imshow(result["coherence"], cmap="viridis", vmin=0, vmax=1)
axes[3].set_title("Coherence" if LANGUAGE == "en" else "Когерентность")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()

## Noise and integration scale

In [ ]:
noise_levels = [0.02, 0.08, 0.14]
scales = [1.0, 3.0, 6.0]
error_table = np.zeros((len(noise_levels), len(scales)))
for row, noise in enumerate(noise_levels):
    noisy_image, noisy_truth = synthetic_parallel_fibers(
        angle_degrees=-25.0,
        noise_std=noise,
        seed=8,
    )
    evaluation_mask = np.zeros(noisy_image.shape, dtype=bool)
    evaluation_mask[10:-10, 10:-10] = True
    for column, scale in enumerate(scales):
        estimate = structure_tensor_orientation(
            noisy_image,
            StructureTensorParameters(1.0, scale),
        )
        error_table[row, column] = orientation_error_metrics(
            estimate["orientation"],
            noisy_truth,
            evaluation_mask,
        )["mae_deg"]
error_table

## Curved field

In [ ]:
curved_image, curved_truth = synthetic_curved_fibers(noise_std=0.05, seed=9)
curved_result = structure_tensor_orientation(
    curved_image,
    StructureTensorParameters(1.0, 3.0),
)
curved_mask = confidence_mask(curved_result, 0.35, 0.1, border=10)
orientation_error_metrics(curved_result["orientation"], curved_truth, curved_mask)

## Crossing fibers

In [ ]:
crossing_image, true_families = synthetic_crossing_fibers()
crossing_result = structure_tensor_orientation(
    crossing_image,
    StructureTensorParameters(1.0, 4.0),
)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(crossing_image, cmap="gray")
axes[1].imshow(np.rad2deg(crossing_result["orientation"]), cmap="twilight", vmin=-90, vmax=90)
axes[2].imshow(crossing_result["coherence"], cmap="viridis", vmin=0, vmax=1)
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
print("True families, degrees:", np.rad2deg(true_families))

## Axial summary

In [ ]:
mean_angle, order = weighted_axial_mean(
    result["orientation"][mask],
    result["coherence"][mask],
)
print("Mean axial angle, degrees:", np.rad2deg(mean_angle))
print("Order parameter:", order)
print("Accepted-pixel coverage:", mask.mean())